[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 困难：分组查询注意力（Grouped Query Attention, GQA）

实现 **Grouped Query Attention** —— 该机制被 LLaMA 2、Mistral 等模型采用，用于大幅减少 KV Cache 大小。

与多头注意力（MHA）类似，但 **KV 的头数少于 Q 的头数**。每组 Query 头共享同一个 Key/Value 头。

### 函数签名
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # 自注意力
```

### 实现要求
- `self.W_q`: `nn.Linear(d_model, d_model)` —— 完整的 Q 投影
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` —— 减少后的 K 投影
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` —— 减少后的 V 投影
- `self.W_o`: `nn.Linear(d_model, d_model)` —— 输出投影
- `d_k = d_model // num_heads`
- 使用 `repeat_interleave` 将 KV 头扩展到与 Q 头数量一致
- 当 `num_kv_heads == num_heads` 时，应与标准多头注意力（MHA）行为完全一致

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

1. W_q 查询权重矩阵和 MHA 一致， W_k 和 W_v 键权重矩阵和值权重矩阵维度设置为 num_kv_heads * d_k
2. 根据 x 投影层 Q, K, V
3. 计算维度拓展倍速 repeat_number = num_heads // num_kv_heads
4. 使用 Tensor.repeat_interleave(repeat_number, dim=1) 将 K, V 转成相应的 num_heads 维度
5. 然后和 MHA 一样计算 Attention

注意:
tensor.repeat 是将整个序列当作整体复制，而 tensor.repeat_interleave 是让每个元素原地重复后再下一个, 如果不指定维度，则将 tensor 展平成一维

### `view()` vs `reshape()` 对比

| 特性                  | `tensor.view()`                              | `tensor.reshape()`                              |
|-----------------------|----------------------------------------------|-------------------------------------------------|
| **内存连续性要求**    | 必须连续（否则报错）                         | 不要求连续                                      |
| **内存共享**          | 始终共享（只改 view，不会 copy）             | **优先**共享，必要时会自动 copy                 |
| **底层实现**          | 纯粹修改 metadata                            | 内部会判断，能 view 就 view，不能就 clone + view |
| **最推荐的使用场景**  | 已知张量是连续的 & 追求最高性能, 为了省显存/内存              | 99% 的日常使用场景（更安全、更不容易出错）      |


In [ ]:
class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # d_model: 模型维度
        # num_heads: Q 的头数 (完整头数)
        # num_kv_heads: KV 的头数 (减少后的头数)
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        
        # 每个头的维度
        self.d_k = d_model // num_heads
        
        # 1. 初始化投影层 W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # x -> Q, K, V -> 
        batch, seq_len, _ = x.shape
        
        # 2. 线性投影得到 Q, K, V
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)  # (batch, seq_len, num_kv_heads * d_k)
        V = self.W_v(x)  # (batch, seq_len, num_kv_heads * d_k)
        
        # 3. 将 Q, K, V 重塑为多头形式
        # Q: (batch, seq_len, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.reshape(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        # K, V: (batch, seq_len, num_kv_heads, d_k) -> (batch, num_kv_heads, seq_len, d_k)
        K = K.reshape(batch, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2)
        V = V.reshape(batch, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2)
        
        # 4. 扩展 KV 头数以匹配 Q 头数
        # 每组 Query 头共享同一个 Key/Value 头
        # 例如: num_heads=8, num_kv_heads=2，则每 4 个 Q 头共享 1 个 KV 头
        repeat_factor = self.num_heads // self.num_kv_heads
        # 使用 repeat_interleave 扩展: (batch, num_kv_heads, ...) -> (batch, num_heads, ...)
        K = K.repeat_interleave(repeat_factor, dim=1)
        V = V.repeat_interleave(repeat_factor, dim=1)
        
        # 5. 计算缩放点积注意力
        # 公式: scores = Q @ K.T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # softmax 得到注意力权重
        attn_weights = torch.softmax(scores, dim=-1)
        # 输出: (batch, num_heads, seq_len, d_k)
        out = torch.matmul(attn_weights, V)
        
        # 6. 拼接多头输出
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads, d_k)
        out = out.transpose(1, 2).contiguous()
        # (batch, seq_len, num_heads, d_k) -> (batch, seq_len, d_model)
        out = out.reshape(batch, seq_len, self.d_model)
        
        # 7. 最终输出投影
        out = self.W_o(out)
        
        return out

In [ ]:
# 🧪 调试测试
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q 形状:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k 形状:", gqa.W_k.weight.shape)  # (8, 32)  — 仅 2 个 KV 头 * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("输出形状:", out.shape)           # (2, 6, 32)

In [ ]:
from torch_judge import check
check('gqa')